# document_list

> Document list with virtual collection, keyboard navigation, and bulk selection

In [ ]:
#| default_exp components.document_list

In [ ]:
#| export
from typing import Any, Callable, List, Set

from fasthtml.common import (
    Div, Span, A, Button, Input, Label, Script
)

# DaisyUI components
from cjm_fasthtml_daisyui.components.actions.button import (
    btn, btn_colors, btn_styles, btn_sizes
)
from cjm_fasthtml_daisyui.components.data_input.checkbox import (
    checkbox, checkbox_sizes
)
from cjm_fasthtml_daisyui.utilities.semantic_colors import text_dui

# Tailwind utilities
from cjm_fasthtml_tailwind.utilities.spacing import p, m
from cjm_fasthtml_tailwind.utilities.sizing import w, min_h
from cjm_fasthtml_tailwind.utilities.typography import font_size, font_weight
from cjm_fasthtml_tailwind.utilities.layout import overflow
from cjm_fasthtml_tailwind.utilities.flexbox_and_grid import (
    flex_display, flex_direction, items, justify, gap, grow
)
from cjm_fasthtml_tailwind.core.base import combine_classes

# Icons
from cjm_fasthtml_lucide_icons.factory import lucide_icon

# Design system recipes (V11 icon-size roles)
from cjm_fasthtml_design_system.icons import icons

# Virtual Collection
from cjm_fasthtml_virtual_collection.core.models import (
    VirtualCollectionConfig, VirtualCollectionState, ColumnDef, CellRenderContext,
    VirtualCollectionUrls,
)
from cjm_fasthtml_virtual_collection.core.html_ids import VirtualCollectionHtmlIds
from cjm_fasthtml_virtual_collection.core.button_ids import VirtualCollectionButtonIds
from cjm_fasthtml_virtual_collection.components.collection import render_virtual_collection

# Keyboard Navigation
from cjm_fasthtml_keyboard_navigation.components.system import render_keyboard_system
from cjm_fasthtml_keyboard_navigation.core.manager import ZoneManager
from cjm_fasthtml_virtual_collection.keyboard.actions import (
    create_collection_focus_zone,
    create_collection_nav_actions,
    build_collection_url_map,
    apply_nav_sync,
)

# JS generators
from cjm_fasthtml_virtual_collection.js.scroll import generate_scroll_nav_js
from cjm_fasthtml_virtual_collection.js.scrollbar import generate_scrollbar_js
from cjm_fasthtml_virtual_collection.js.auto_fit import generate_auto_fit_js, auto_fit_callback_name

# Viewport fit
from cjm_fasthtml_viewport_fit.models import ViewportFitConfig
from cjm_fasthtml_viewport_fit.components import render_viewport_fit_script

# Local imports
from cjm_transcript_workflow_management.models import DocumentSummary, ManagementUrls
from cjm_transcript_workflow_management.html_ids import ManagementHtmlIds
from cjm_transcript_workflow_management.utils import format_duration, format_date
from cjm_transcript_workflow_management.components.helpers import (
    render_icon_button, render_media_type_badge, render_empty_state,
    render_delete_modal, DEBUG_MANAGEMENT_RENDER
)

## Column Definitions

Define the virtual collection columns for the document list table.

In [ ]:
#| export
def build_document_columns() -> tuple:
    """Column definitions for the document list virtual collection."""
    return (
        ColumnDef(key="select", header="", sortable=False, header_cls=str(w("10"))),
        ColumnDef(key="title", header="Title", sortable=True),
        ColumnDef(key="type", header="Type", sortable=False),
        ColumnDef(key="segments", header="Segments", sortable=True),
        ColumnDef(key="duration", header="Duration", sortable=True),
        ColumnDef(key="created", header="Created", sortable=True),
        ColumnDef(key="actions", header="Actions", sortable=False),
    )

In [ ]:
cols = build_document_columns()
assert len(cols) == 7
assert cols[0].key == "select"
assert cols[1].sortable == True
assert cols[2].sortable == False
print(f"Columns: {[c.key for c in cols]}")

## Cell Renderer

Factory that creates a cell renderer callback for the virtual collection. Each column key maps to domain-specific rendering logic.

In [ ]:
#| export
def create_document_cell_renderer(
    get_selected:Callable,  # () -> Set[str] of selected document IDs
    urls:'ManagementUrls',  # URL bundle for action buttons
    toggle_select_url:str="",  # URL for checkbox toggle route
) -> Callable:  # render_cell(item, ctx) -> FT component
    """Create a cell renderer for the document list virtual collection."""
    
    def render_cell(item, ctx):
        doc = item
        key = ctx.column.key
        
        if key == "select":
            checked = doc.document_id in get_selected()
            return Input(
                type="checkbox",
                cls=combine_classes(checkbox, checkbox_sizes.sm),
                checked=checked,
                hx_post=toggle_select_url,
                hx_vals=f'{{"doc_id": "{doc.document_id}"}}',
                hx_swap="none",
                onclick="event.stopPropagation()",
            )
        elif key == "title":
            return Span(doc.title, cls=str(font_weight.medium))
        elif key == "type":
            return render_media_type_badge(doc.media_type)
        elif key == "segments":
            return Span(f"{doc.segment_count} segs")
        elif key == "duration":
            return Span(format_duration(doc.total_duration))
        elif key == "created":
            return Span(format_date(doc.created_at), cls=str(font_size.sm))
        elif key == "actions":
            return Div(
                render_icon_button(
                    "eye", "View",
                    size=str(btn_sizes.sm),
                    hx_get=f"{urls.document_detail}?doc_id={doc.document_id}",
                    hx_target=f"#{ManagementHtmlIds.PAGE_CONTENT}",
                    hx_swap="innerHTML",
                ),
                A(
                    lucide_icon("download", size=icons.icon_button),
                    title="Export",
                    cls=combine_classes(btn, btn_styles.ghost, btn_sizes.sm),
                    href=f"{urls.export_document}?doc_id={doc.document_id}",
                    download=True,
                ),
                render_icon_button(
                    "trash-2", "Delete",
                    color=str(btn_colors.error),
                    size=str(btn_sizes.sm),
                    onclick=(
                        f"prepareDeleteModal('{doc.document_id}', '{doc.title}', {doc.segment_count}); "
                        f"document.getElementById('{ManagementHtmlIds.DELETE_MODAL}').showModal()"
                    ),
                ),
                cls=combine_classes(flex_display, gap(1)),
            )
        return Span("")
    
    return render_cell

In [ ]:
from fasthtml.common import to_xml

urls = ManagementUrls(
    management_page="/manage/documents/management_page",
    list_documents="/manage/documents/list_documents",
    document_detail="/manage/documents/document_detail",
    delete_document="/manage/documents/delete_document",
    delete_selected="/manage/documents/delete_selected",
    export_document="/manage/export/export_document",
    export_all="/manage/export/export_all",
    import_graph="/manage/import/import_graph",
)

selected = {"abc-123"}
renderer = create_document_cell_renderer(lambda: selected, urls, "/toggle")

doc = DocumentSummary("abc-123", "1. Laying Plans", "audio", 247, 2537.0, 1740000000.0)
ctx_title = CellRenderContext(row_index=0, column=ColumnDef(key="title", header="Title"), total_items=1)
title_cell = renderer(doc, ctx_title)
assert "1. Laying Plans" in to_xml(title_cell)

ctx_select = CellRenderContext(row_index=0, column=ColumnDef(key="select", header=""), total_items=1)
select_cell = renderer(doc, ctx_select)
assert "checked" in to_xml(select_cell)
print("Cell renderer OK")

## Toolbar

Select All checkbox (HTMX-driven) and bulk action buttons. Server-side selection state replaces client-side checkbox management.

In [ ]:
#| export
def render_toolbar(
    urls:ManagementUrls,  # URL bundle for route endpoints
    selected_count:int=0,  # Number of selected documents
    total_count:int=0,  # Total number of documents
    select_all_url:str="",  # URL for select-all toggle route
) -> Any:  # Toolbar element
    """Render the document list toolbar with Select All and bulk actions."""
    all_selected = selected_count == total_count and total_count > 0
    
    return Div(
        # Left side: Select All
        Label(
            Input(
                type="checkbox",
                cls=combine_classes(checkbox, checkbox_sizes.sm),
                id=ManagementHtmlIds.SELECT_ALL,
                checked=all_selected,
                hx_post=select_all_url,
                hx_swap="none",
            ),
            Span("Select All", cls=combine_classes(font_size.sm, m.l(2))),
            cls=combine_classes(flex_display, items.center),
        ),
        # Right side: Delete Selected
        Button(
            lucide_icon("trash-2", size=icons.text_button),
            f"Delete Selected ({selected_count})" if selected_count else "Delete Selected",
            cls=combine_classes(
                btn, btn_colors.error, btn_styles.outline, btn_sizes.sm,
                flex_display, items.center, gap(1)
            ),
            id=ManagementHtmlIds.DELETE_SELECTED_BTN,
            disabled=selected_count == 0,
            onclick=f"prepareBulkDeleteModal(); document.getElementById('{ManagementHtmlIds.BULK_DELETE_MODAL}').showModal()",
        ),
        id=ManagementHtmlIds.TOOLBAR,
        cls=combine_classes(flex_display, justify.between, items.center, m.b(2)),
    )

In [ ]:
toolbar = render_toolbar(urls, selected_count=2, total_count=5, select_all_url="/select_all")
xml = to_xml(toolbar)
assert ManagementHtmlIds.TOOLBAR in xml
assert "Delete Selected (2)" in xml
assert "disabled" not in xml  # Button should be enabled with 2 selected
print("Toolbar OK")

toolbar_none = render_toolbar(urls, selected_count=0, total_count=5, select_all_url="/select_all")
xml_none = to_xml(toolbar_none)
assert "disabled" in xml_none  # Button disabled with 0 selected
print("Toolbar disabled OK")

## Client-Side JavaScript

Checkbox selection management and delete modal preparation.

In [ ]:
#| export
def render_list_scripts(
    urls:ManagementUrls,  # URL bundle for route endpoints
) -> Any:  # Script element
    """Render client-side JavaScript for delete modal management."""
    return Script(f"""
    function prepareDeleteModal(docId, title, segCount) {{
        var body = document.getElementById('{ManagementHtmlIds.DELETE_MODAL_BODY}');
        if (body) {{
            body.innerHTML = '<p>Permanently delete "' + title + '" and all ' + segCount + ' segments?</p>'
                + '<p class="{combine_classes(font_size.sm, text_dui.base_content.opacity(60), m.t(2))}">This action cannot be undone.</p>';
        }}
        var confirmBtn = document.querySelector('#{ManagementHtmlIds.DELETE_MODAL} [data-delete-confirm]');
        if (confirmBtn) {{
            confirmBtn.setAttribute('hx-post', '{urls.delete_document}');
            confirmBtn.setAttribute('hx-vals', JSON.stringify({{doc_id: docId}}));
            confirmBtn.setAttribute('hx-swap', 'none');
            htmx.process(confirmBtn);
        }}
    }}

    function prepareBulkDeleteModal() {{
        var body = document.getElementById('{ManagementHtmlIds.BULK_DELETE_MODAL_BODY}');
        if (body) {{
            body.innerHTML = '<p>Permanently delete selected documents and all their segments?</p>'
                + '<p class="{combine_classes(font_size.sm, text_dui.base_content.opacity(60), m.t(2))}">This action cannot be undone.</p>';
        }}
    }}
    """)

In [ ]:
scripts = render_list_scripts(urls)
xml = to_xml(scripts)
assert "prepareDeleteModal" in xml
assert "prepareBulkDeleteModal" in xml
assert "toggleSelectAll" not in xml  # Removed — now server-side
print("Scripts OK")

## Document List Component

Complete document list with virtual collection, keyboard navigation, toolbar, modals, and scripts. Replaces the previous static HTML table.

In [ ]:
#| export
def render_document_list(
    items:list,  # Current document list (List[DocumentSummary])
    selected:set,  # Selected document IDs (Set[str])
    vc_config:'VirtualCollectionConfig',  # VC configuration
    vc_state:'VirtualCollectionState',  # VC state
    vc_ids:'VirtualCollectionHtmlIds',  # VC HTML IDs
    vc_btn_ids:'VirtualCollectionButtonIds',  # VC button IDs
    vc_urls:'VirtualCollectionUrls',  # VC route URLs
    mgmt_urls:'ManagementUrls',  # Management URLs for actions
    render_cell:Callable,  # Cell renderer callback
    select_all_url:str="",  # URL for select-all toggle route
) -> Any:  # Complete document list component
    """Render the document list with virtual collection, keyboard nav, and modals."""
    if DEBUG_MANAGEMENT_RENDER:
        print(f"[RENDER] document_list: {len(items)} docs, {len(selected)} selected")
    
    # Empty state
    if not items:
        return Div(
            render_empty_state(),
            id=ManagementHtmlIds.DOCUMENT_LIST,
        )
    
    # Keyboard system — returns a dataclass, unpack individual components
    zone = create_collection_focus_zone(vc_ids)
    nav_actions = create_collection_nav_actions(zone.id, vc_btn_ids)
    manager = ZoneManager(zones=(zone,), actions=nav_actions)
    url_map = build_collection_url_map(vc_btn_ids, vc_urls)
    target_map = {btn_id: f"#{vc_ids.rows}" for btn_id in url_map}
    swap_map = {btn_id: "none" for btn_id in url_map}
    
    kb_system = render_keyboard_system(
        manager,
        url_map=url_map,
        target_map=target_map,
        swap_map=swap_map,
    )
    apply_nav_sync(kb_system, vc_ids)
    
    # Viewport fit for auto-sizing
    vf_config = ViewportFitConfig(
        namespace=vc_config.prefix,
        target_id=vc_ids.wrapper,
        resize_callback=auto_fit_callback_name(vc_config),
        enable_htmx_settle=False,
    )
    
    # JS scripts for VC navigation
    js_code = "\n".join([
        generate_scroll_nav_js(vc_ids, vc_btn_ids),
        generate_scrollbar_js(vc_ids, vc_urls),
        generate_auto_fit_js(vc_ids, vc_config, vc_urls, total_items=len(items)),
    ])
    
    return Div(
        # Toolbar
        render_toolbar(mgmt_urls, len(selected), len(items), select_all_url),
        # Virtual collection wrapped in flex container
        Div(
            render_virtual_collection(
                items=items,
                config=vc_config,
                state=vc_state,
                ids=vc_ids,
                urls=vc_urls,
                render_cell=render_cell,
                render_empty=render_empty_state,
            ),
            cls=combine_classes(
                flex_display, flex_direction.col, grow(1), min_h(0),
                overflow.hidden,
            ),
        ),
        # Alert area for feedback messages
        Div(id=ManagementHtmlIds.ALERT_AREA),
        # Single delete confirmation modal
        render_delete_modal(
            modal_id=ManagementHtmlIds.DELETE_MODAL,
            body_id=ManagementHtmlIds.DELETE_MODAL_BODY,
            title="Delete Document?",
            confirm_attrs={
                "data_delete_confirm": "true",
                "onclick": f"document.getElementById('{ManagementHtmlIds.DELETE_MODAL}').close()",
            },
        ),
        # Bulk delete confirmation modal (swap=none — response is all OOB)
        render_delete_modal(
            modal_id=ManagementHtmlIds.BULK_DELETE_MODAL,
            body_id=ManagementHtmlIds.BULK_DELETE_MODAL_BODY,
            title="Delete Selected Documents?",
            confirm_attrs={
                "hx_post": mgmt_urls.delete_selected,
                "hx_swap": "none",
                "onclick": f"document.getElementById('{ManagementHtmlIds.BULK_DELETE_MODAL}').close()",
            },
        ),
        # Keyboard system components (unpacked from dataclass)
        kb_system.script,
        kb_system.hidden_inputs,
        kb_system.action_buttons,
        # Scripts
        render_list_scripts(mgmt_urls),
        Script(js_code),
        render_viewport_fit_script(vf_config),
        id=ManagementHtmlIds.DOCUMENT_LIST,
        cls=combine_classes(flex_display, flex_direction.col, grow(1), min_h(0)),
    )

In [ ]:
# Minimal render test — verify structure without running a server
# Full integration tests happen via the demo app
assert render_document_list is not None
assert build_document_columns is not None
assert create_document_cell_renderer is not None
assert render_toolbar is not None
assert render_list_scripts is not None
print("document_list exports OK")

In [ ]:
# Test empty state still works with new signature
vc_config = VirtualCollectionConfig(prefix="test", columns=build_document_columns())
vc_state = VirtualCollectionState()
vc_ids = VirtualCollectionHtmlIds(prefix="test")
vc_btn_ids = VirtualCollectionButtonIds(prefix="test")
vc_urls = VirtualCollectionUrls()

empty_list = render_document_list(
    items=[], selected=set(),
    vc_config=vc_config, vc_state=vc_state,
    vc_ids=vc_ids, vc_btn_ids=vc_btn_ids,
    vc_urls=vc_urls, mgmt_urls=urls,
    render_cell=lambda i, c: Span(""),
    select_all_url="/select_all",
)
xml = to_xml(empty_list)
assert ManagementHtmlIds.DOCUMENT_LIST in xml
assert "No documents found." in xml
print("Empty state OK")

In [ ]:
#| hide
import nbdev; nbdev.nbdev_export()